# DATA PREPROCESSING

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# Load dataset
churn_df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")


### i) Separate target and predictors

In [2]:
churn_df = churn_df.drop(columns=["customerID"])
churn_df["SeniorCitizen"] = churn_df["SeniorCitizen"].astype("object")

x = churn_df.drop(columns=["Churn"])

y = churn_df["Churn"].map({"Yes": 1,"No": 0})

print("X shape:", x.shape)
print("y shape:", y.shape)

print(y.value_counts())

X shape: (7043, 19)
y shape: (7043,)
Churn
0    5174
1    1869
Name: count, dtype: int64


### ii) Convert 'TotalCharges' to numerical

In [3]:
x["TotalCharges"] = pd.to_numeric(
    x["TotalCharges"],
    errors="coerce"
)

x["TotalCharges"].isna().sum()

#The blank values will become missing values (NaN) and will be handled by our imputation pipeline.

np.int64(11)

### iii) Identify numerical and categorical variables

In [4]:
numerical_features = x.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = x.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("Number of numerical features:", len(numerical_features))

print("\nCategorical features:", categorical_features)
print("Number of categorical features:", len(categorical_features))

Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Number of numerical features: 3

Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Number of categorical features: 16


C:\Users\hp\AppData\Local\Temp\ipykernel_16860\2428020798.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(


### iv) Train-test split

In [5]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", x_train.shape)
print("Testing set:", x_test.shape)

# Churn is an imbalanced classification problem. Approximately 26.5% of customers churn.
#Stratification (stratify=y) maintains approximately the same class proportions in both training and testing datasets.

Training set: (5634, 19)
Testing set: (1409, 19)


### v) Build numerical and categorical pipelines

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (OneHotEncoder)

from sklearn.impute import SimpleImputer

In [15]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Scaling is not necessary for Random Forest or XGBoost because both are tree-based algorithms.

In [16]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [17]:
# Combine both pipelines using the ColumnTransformer. 
# The ColumnTransformer applies the numerical pipeline to the numerical features and the categorical pipeline to the categorical features.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [18]:
# Verify the preprocessor.

x_train_processed = preprocessor.fit_transform(x_train)

print("Original training shape:", x_train.shape)

print("Processed training shape:",
      x_train_processed.shape)

Original training shape: (5634, 19)
Processed training shape: (5634, 46)


### vi) Check for data leakage

We should confirm that:
1. customerID is not among our predictors.
2. Churn is not among our predictors.
3. The test data was not used to fit the imputer, scaler or encoder.
4. All preprocessing transformations are fitted using training data only.

In [19]:
assert "customerID" not in x.columns
assert "Churn" not in x.columns

assert len(x_train) + len(x_test) == len(x)

assert y_train.index.equals(x_train.index)
assert y_test.index.equals(x_test.index)

print("All preprocessing integrity checks passed.")

All preprocessing integrity checks passed.


In [20]:
#Establish baseline distribution of the target variable (Churn) in the training dataset.

print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(3))

Training target distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Training target proportions:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [24]:
import os

os.makedirs("../data/processed", exist_ok=True)

cleaned_df = x.copy()

cleaned_df["Churn"] = y

cleaned_df.to_csv(
    "../data/processed/cleaned_telco_churn.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


### 1. Random Forest

### 2. XGBoost

In [ ]:
xgb_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),

    ("smote", SMOTE(
        random_state=42,
        sampling_strategy="auto"
    )),

    ("classifier", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

# NOTE: XGBoost uses gradient boosting, building trees sequentially to correct errors made by previous trees.